In [1]:
!pip uninstall -y torchvision -q
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 133.1 MB/s eta 0:00:00


In [2]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Transformers: 5.19.0
CUDA: True
GPU: NVIDIA A100-SXM4-40GB


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)
model.eval()

print("Model yüklendi.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model yüklendi.


In [4]:
def generate_text(
    prompt,
    temperature=0.7,
    top_k=20,
    top_p=0.8,
    repetition_penalty=1.1,
    max_new_tokens=150
):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful technical assistant. "
                "Answer clearly, accurately and concisely."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,

            do_sample=True,

            temperature=temperature,
            top_k=top_k,
            top_p=top_p,

            repetition_penalty=repetition_penalty
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[-1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer

In [5]:
prompt = """
Elektrik motorlarında rulman arızasının
üç yaygın nedenini açıkla.
"""

print(
    generate_text(
        prompt,
        temperature=0.2
    )
)

Elektrik motorlarında rulman arızasının üç yaygın nedeni:

1. Kullanım ve kullanım durumunda:
   - Elektrik motorları kullanırken, rulman arızası önemlidir.

2. Teknoloji ve teknik:
   - Elektronik motorların rulman arızasını temsil eden teknolojilerin kullanımı.

3. Uygulamaların genel özellikleri:
   - Elektrik motorları kullanırken, rulman arızası önemlidir.

4. Örneği:
   - Motorlar, rastgele hızda hızlanır ve bu hızın rulman ar


In [6]:
print(
    generate_text(
        prompt,
        temperature=0.7
    )
)

Elektrik motorlarında rulman arızası üç yaygın nedeni:

1. Fiziksel durumu: Motorun üstünde güç tüketilmek için gerekli rulmalar yaratabilir.

2. Enerji kaybı: Motorun üstünde yeterli yolu sağlayabilen en büyük enerji kaynağı olmalıdır.

3. Kullanım alanları: Motorun çeşitli alanlarda kullanıldığından, rulman arızaları zayıflamaz.


In [7]:
print(
    generate_text(
        prompt,
        temperature=1.3
    )
)

Elektro motores sırasında rulman ara sıra,
çok çeşitli etkinliklere olanak sağlar:

1. Kondensasyon: Elektro motornun kalp içindeki karmaşık elektromekanisme, bu konuda rulmalar sürekli olarak kendi kordonu ile karşılaştırdığı anlam.

2. Tekrarlı çalışmayı sağlıyor: Konsistent bir rütbelim yaparak işlerle ilerleyen rulamalar için kullanılan özellik.

3. Düzgün hizmet uygulanabilirliği sağlanıyor: Rözge ve yoluyla farklı konuslarına yönelik rülmenler oluşturulurken.

4.


In [8]:
print(
    generate_text(
        prompt,
        temperature=0.7,
        top_k=5,
        top_p=1.0
    )
)

Rulman arızası elektrik motorlarında yaygın olarak kullanılan üç önemli nedeni:

1. Tekrarlı çalışması: Enerji kaynağıyla uygun olan rulman arızası kullanılır.

2. Yüksek enerjilerde kullanımı: Enerji kaynakları ve motorun üstündeki güç kaynağı ile yüksek enerjiler üzerinde çalışır.

3. Sosyal ve teknolojik faydalar: Rulman arızalarda daha fazla sosyo-teknolojik özellikler (yani daha hızlı, daha efektif) kullanılabilir.

4. Uygulama için gereken faktöre bağlıdır: Motorların kullanımında çeşitli faktörleri


In [9]:
print(
    generate_text(
        prompt,
        temperature=0.7,
        top_k=20,
        top_p=1.0
    )
)

Elektrik motorlarında rulman arızasının üç yaygın noktalıdır:

1. Kuvvet: Motorların hızını ve hızınızıza uygun bir şekilde ayarlanmayı amaçlayan araçlar.

2. Sıfırlık: Rastgele hızda hareket eden motorler.

3. Yolluk: Motorları kullanırken veya sonrasında kullanılması gereken araçlar.


In [10]:
print(
    generate_text(
        prompt,
        temperature=0.7,
        top_k=100,
        top_p=1.0
    )
)

1. Elektrik motorlarında rulman arızası genellikle 50 Hz ve 60 Hz'dir.

2. Bu faydalı faktöre ulaşmak için kendi elektrikli komüniterine veya gecen maddeleri dahil olmalıdır.

3. Bir daha az teşviklik olan motor 50 Hz'yi kullanılırken, yüksek bir hızda güç sağlamak için 60 Hz'in kullanılmasını önermektedir.

4. 50 Hz'unun hedef degralı değer 78,4 rad/s olduğu için, 电机ler arasında 30 Hz'dan daha düşük oranda daha hızlı güç al


In [11]:
print(
    generate_text(
        prompt,
        temperature=0.7,
        top_k=0,
        top_p=0.5
    )
)

Elektrik motorlarında rulman arızasının üç yaygın nedeni:

1. Körünüz: Elektromekanik sistemlerde, elektrikli rulman arızası genellikle körünüz olarak tanımlanır.

2. Rastgele hız: Elektromekanik sistemlerde, rastgele hızın etkisi arızasını belirleyen bir nedenidir.

3. Sistemin durumu: Elektromekanik sisteminin durumunu etkileyen bir nedenidir.

4. Tekniksel bir faydalı: Elektromekanik sistem


In [12]:
print(
    generate_text(
        prompt,
        temperature=0.7,
        top_k=0,
        top_p=0.8
    )
)

Elektrik motorları arasında rulman arızası üç yaygın nedeni:

1. Lük ve iyi rastgele kontrol: Elektro-optik sistemlerin etkinliği artırmak için çeşitli yöntemleri kullanılır.

2. Yüksek durumlara erişim sağlama: Motorun yüksek durumlarda en iyisi çalışması için güvenli sistemler kullanılabilir.

3. Hızlı ve hızlı yürüyüşsü sağlamak: Motorların hızla hareket edilmesi için daha küçük ve daha uzun sürmüş sistemin alınması önemlidir.


In [13]:
print(
    generate_text(
        prompt,
        temperature=0.7,
        top_k=0,
        top_p=0.95
    )
)

Elektrik motorlarına rülman arızası için kullanılırken:

1. Çıkmalı: Kullanılabilirlerin belirli zaman içinde dönmüş olup bu durumda rülme izin verilmez.

2. Dizel Motor: Bu motor her zaman dört dize değişiyor ve rülmen iki dizi genellikle kullanılabilir.

3. Marke Yeni: Eğer motorunun marca yeni olduğunu belirtmek isterse, rülman arızasını içerir.

4. Seviyede: Rülman arızayı kullanıcılar tarafından ayarlanır.

5. Eklentiler: Yüksek volume


In [14]:
print(
    generate_text(
        prompt,
        repetition_penalty=1.0
    )
)

Elektrik motorlarında rulman arızasının üç yaygın nedeni:

1. Enerji kaynaklığı: Elektrik motorlar, enerji kaynakları tarafından kullanılabilir ve rastgele olarak kullanılır.

2. Fiziksel özellikler: Motorların enerji kaynaklarını kullanarak hızı ve hızla güç kaybı alır.

3. Tekniksel özellikler: Motorların motor kuvvetlerini, motorunun yoluyla güç alması için kullanılarak çalışır.

4. Tekniksel özellikler: Motorların motor kuvvetlerini, motorunun yoluyla güç alması için kullanılarak çalışır.




In [15]:
print(
    generate_text(
        prompt,
        repetition_penalty=1.2
    )
)

Elektrokardiya (EC) rülme arzını çözerken:

1. Rüşvenin hafifliği artırmak için kullanılan teknolojisi.

2. Elektrodeyi kalmayıza uygun bir fırças veya ekip para kullanarak elektrokardiye göndermesi.

3. Elektrodeki aktörlerinin güç tüketimiyle ilgili faktörler.

4. Örneğin, elektronik sistemlerinde daha fazlasına ihtiyaç duyarlı olabilir.

5. Elektrosistemleri kontrol etmek için gereklidir.

6. Genellikle elektrikli sistemlerden elde


In [16]:
def generate_greedy(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    generated = outputs[0][
        inputs["input_ids"].shape[-1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [17]:
for i in range(3):

    print("DENEME:", i + 1)

    print(
        generate_greedy(
            "Bir robot gelecekte hangi işleri yapabilir?"
        )
    )

    print("-" * 80)

DENEME: 1
Bir robot geleceğe göre, çeşitli işler ve görevlileri vardır:

1. Uygulamalar: Robotlar genellikle uygulamalar yapmaya çalışır. Bu uygulamalar, birlikte diğer sistemlerin veya kullanıcıların işlemlerini düzenleyebilir.

2. Sosyal bilgiler: Robotlar sosyal bilgileri oluşturur ve paylaşır. Bu bilgiler, kullanıcıların
--------------------------------------------------------------------------------
DENEME: 2
Bir robot geleceğe göre, çeşitli işler ve görevlileri vardır:

1. Uygulamalar: Robotlar genellikle uygulamalar yapmaya çalışır. Bu uygulamalar, birlikte diğer sistemlerin veya kullanıcıların işlemlerini düzenleyebilir.

2. Sosyal bilgiler: Robotlar sosyal bilgileri oluşturur ve paylaşır. Bu bilgiler, kullanıcıların
--------------------------------------------------------------------------------
DENEME: 3
Bir robot geleceğe göre, çeşitli işler ve görevlileri vardır:

1. Uygulamalar: Robotlar genellikle uygulamalar yapmaya çalışır. Bu uygulamalar, birlikte diğer sistemlerin vey

In [19]:
for i in range(3):

    print("DENEME:", i + 1)

    print(
        generate_text(
            "Bir robot gelecekte hangi işleri yapabilir?",
            temperature=0.9,
            top_k=30,
            top_p=0.9
        )
    )

    print("-" * 80)

DENEME: 1
Robotların gelecekteki bir dizi önemli görevleri vardır:

1. Uygulamak: Robotlar birçok uygulama için kullanılabilir ve teknolojiye uyum sağlar.

2. Etkinlikler: İşlerle ilgili birlikte büyük uzmanları ve teknici kullanılır.

3. İletişim: Eski ve yeni çalışanlara iletişimi verirken derobotlardan destek alınır.

4. Sosyal medya: Mobil telefonu ile iletişim kurarak sosyal medya paylaşılır.

5. Üretmen / Kalitayı değiştirme: Elektronik kaliteyi değiştirerek hedeflere dayalı bir mes
--------------------------------------------------------------------------------
DENEME: 2
Eğer bir robot gelecekte herkesin ihtiyaç duyabileceği işler yapabilen bir araç olarak kullanılırdığınızda, bu şekilde çalışmak isteyeceğiniz bazı işler genellikle aşağıdaki gibi olabilir:

1. **Güvenlik ve güvenliği:** 
   Robotlar, büyük öncelikle küçük bir ülkenizin güvenlik ve güvenlik problemleri için destek gösterebilir.

2. **İşletmeyi ve durum kaynakları yönetmek:**
   Robotler, bir etkinlilik sistemini 

In [20]:
technical = generate_text(
    prompt,
    temperature=0.3,
    top_k=20,
    top_p=0.7,
    repetition_penalty=1.1
)

In [21]:
balanced = generate_text(
    prompt,
    temperature=0.7,
    top_k=20,
    top_p=0.8,
    repetition_penalty=1.1
)

In [22]:
creative = generate_text(
    prompt,
    temperature=1.2,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.05
)

In [23]:
print("TEKNİK")
print("=" * 70)
print(technical)

print("\nDENGELİ")
print("=" * 70)
print(balanced)

print("\nYARATICI")
print("=" * 70)
print(creative)

TEKNİK
Elektrik motorlarında rulman arızasının üç yaygın nedeni:

1. Kullanım ve kullanım alanları: Elektrik motorleri, özellikle elektrikli araçlarda (e.g., elektrikli araçlar, elektrikli araçların kendi motorlarını kullanması) ve elektrikli araçların diğer alanlardaki kullanılmasında sıkça kullanılan.

2. Ölçüm ve enerji tüketimi: Elektrik motorler, özellikle elektrikli araçların kullanımı için daha düşük ömürlük ve enerji tüketiminde önemli bir rol oynar.

3. Teknolojik gelişmeler: Elektrik motorler,

DENGELİ
Elektrik motorlarında rulman arızasının üç yaygın nedeni:

1. Çoklu rastgele röportalet: Bu nedeni genellikle elektronik motorlarda ortaya çıkır.

2. Çoğu rastgele röportalet: Bu nedeni genellikle elektronik motorlarda ortaya çıkır.

3. Çoğu rastgele röportalet ve çoklu rastgele röportalet: Bu nedenleri genellikle bir elektronik motor veya alternatif elektromatik sistemlerde ortaya çıkar.

YARATICI
Elektrik motorları çeşitli önemli seçenekleriyle rulman arızanı:

1. Sınırhızı a